# Clase 6 — Análisis espacial: construir variables territoriales

**Sistemas de Información Geográfica**
Especialización en Ciencias Sociales Computacionales — Universidad Nacional Guillermo Brown

| | |
|---|---|
| **Unidad del programa** | 6 — Análisis espacial |
| **Duración** | 3 horas |
| **Versión** | 2026.1 |
| **Docente** | Renzo Polo |
| **Licencia** | CC BY-SA 4.0 |

---

## 1. La pregunta de hoy

> ### ¿Qué tiene cada barrio alrededor, y a qué distancia?

En las clases anteriores obtuvimos datos (Clase 3), los pusimos en el sistema de coordenadas
correcto (Clase 4) y los representamos (Clase 5). En los tres casos la variable ya existía:
alguien había contado los hogares con NBI y nosotros la mapeábamos.

En esta clase la variable la producimos nosotros.

Una **variable territorial** es un atributo de una unidad del espacio que no figura en ningún
registro y que resulta de relacionar dos capas: la cantidad de farmacias por kilómetro cuadrado
de un barrio, el porcentaje de su superficie que está a menos de 500 metros de un centro de
salud, la distancia hasta el efector más próximo. Ninguna de las tres la mide un organismo; las
tres se calculan.

Trabajamos sobre **Recoleta y Villa Lugano**, los dos barrios que descargamos de OpenStreetMap
en la Clase 3, y sobre los efectores públicos de salud que obtuvimos del geoservicio del IGN en
esa misma clase.

## 2. Objetivos de esta clase

Al terminar, deberías poder:

1. **Geocodificar** un listado de direcciones y evaluar el error de la operación.
2. **Transformar** geometrías —centroide, área de influencia, envolvente, simplificación,
   disolución— y elegir la transformación adecuada a cada propósito.
3. **Superponer** capas: intersección y diferencia, y calcular el porcentaje de superficie
   cubierta.
4. **Relacionar** capas con uniones espaciales 1 a 1 y 1 a muchos, y agregar el resultado.
5. **Medir** distancias y determinar el elemento más cercano.
6. **Componer** una tabla de variables territoriales y exportarla.

Cada operación se verifica con un mapa o un gráfico antes de pasar a la siguiente.

## 3. Material de esta clase

Esta clase no tiene presentación. La teoría está en la sección 4.

| Bloque | Antecedente |
|---|---|
| Geocodificación | Clase 3 — fuentes de datos y sus sesgos |
| Transformación de geometrías | Clase 2 — escala y generalización |
| Áreas de influencia y superposición | Clase 4 — medir exige un CRS proyectado |
| Uniones espaciales | Clase 2 — conteo por unidad y MAUP |
| Densidades | Clase 5 — conteo, porcentaje y densidad |

**Datos.** Los tres provienen de la Clase 3:

| Archivo | Contenido | Origen |
|---|---|---|
| `osm_barrios_limites.gpkg` | Contorno de Recoleta y Villa Lugano | OSM, `geocode_to_gdf` |
| `osm_amenities_barrios.gpkg` | 1.770 equipamientos con etiqueta `amenity` | OSM, `features_from_place` |
| `salud_barrios.gpkg` | Los 13 efectores públicos de salud de esos barrios | IGN, capa `ign:salud_020801` |

El tercero es un recorte de `ign_salud.gpkg`, la capa nacional de 8.311 establecimientos que
descargamos por WFS. El recorte se hizo con la misma unión espacial que veremos en el bloque 9.

---

## 4. El repertorio del análisis espacial

**Conceptos clave.** Una **operación espacial** es un procedimiento que toma una o más
geometrías y devuelve —calculado a partir de su forma y su posición— una geometría nueva, una
relación o una medida. Casi todo el análisis espacial vectorial se compone de cuatro familias:

| Familia | Devuelve | Operaciones | Bloque |
|---|---|---|---|
| **Transformar** | Una geometría nueva | `centroid`, `buffer`, `convex_hull`, `envelope`, `simplify`, `union_all` | 7 |
| **Superponer** | La parte común, o la diferencia, entre dos capas | `intersection`, `difference`, `union` | 8 |
| **Relacionar** | Pares de registros de dos capas | `sjoin` con `within`, `intersects`, `contains` | 9 |
| **Medir** | Un número | `area`, `length`, `distance`, `sjoin_nearest` | 10 |

**El producto.** Cada bloque agrega una columna a una tabla de esta forma:

| barrio | superficie_km2 | farmacias_por_km2 | cobertura_500m_perc | dist_salud_m |
|---|---|---|---|---|
| Recoleta | … | … | … | … |
| Villa Lugano | … | … | … | … |

Una fila por unidad de análisis, una columna por variable, y la unidad de medida en el nombre
de la columna. Es el formato que exige el trabajo final.

**Una advertencia sobre unidades.** Superficies, distancias y áreas de influencia son
mediciones, y en coordenadas geográficas (EPSG:4326) la unidad es el grado, que no es una
unidad de longitud. `buffer(500)` sobre datos en EPSG:4326 no produce 500 metros. Por eso, al
preparar los datos, reproyectamos una vez a **EPSG:5347** —POSGAR 2007 faja 5, en metros, la
faja que corresponde a la Ciudad de Buenos Aires— y de ahí en adelante todas las mediciones
usan esas capas.

---

## 5. Preparación

### ▶️ Bibliotecas

In [ ]:
!pip install -q "geopandas==1.0.1" "mapclassify==2.8.1" "matplotlib==3.9.2" \
               "folium==0.17.0" "geopy==2.4.1"

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium

print("Bibliotecas listas")

### ▶️ Capas y sistemas de referencia

Cargamos las tres capas y creamos de una vez sus versiones métricas, con el sufijo `_m`. La
convención se sostiene toda la clase: **`_m` es la capa con la que se mide; la capa sin sufijo
es la que se muestra**, porque los mapas interactivos trabajan en coordenadas geográficas.

In [ ]:
DATOS = "https://raw.githubusercontent.com/renzoepolo/sig-ciencias-sociales/main/datos/"

GEOGRAFICO = "EPSG:4326"   # como vienen los datos, y lo que usan los mapas web
METRICO    = "EPSG:5347"   # POSGAR 2007 faja 5, en metros: para medir

barrios = gpd.read_file(DATOS + "osm_barrios_limites.gpkg")
equip   = gpd.read_file(DATOS + "osm_amenities_barrios.gpkg")
salud   = gpd.read_file(DATOS + "salud_barrios.gpkg")

barrios_m, equip_m, salud_m = (c.to_crs(METRICO) for c in (barrios, equip, salud))

print(f"{len(barrios)} barrios · {len(equip)} equipamientos · {len(salud)} efectores de salud")
print("Medición en:", barrios_m.crs.name, "—", barrios_m.crs.axis_info[0].unit_name)

### 👀 Las tres capas

Antes de operar, verificar que las capas estén donde corresponde. Provienen de dos fuentes
distintas —OSM y el IGN— y un sistema de referencia mal declarado se detecta acá, no tres
bloques más adelante con un resultado inverosímil.

In [ ]:
mapa = barrios.explore(color="grey", style_kwds=dict(fill=False, weight=3),
                       tiles="CartoDB positron", name="Barrios")

equip.explore(m=mapa, color="steelblue", marker_kwds=dict(radius=2),
              tooltip=["name", "amenity"], name="Equipamiento (OSM)")

salud.explore(m=mapa, color="crimson", marker_kwds=dict(radius=6),
              tooltip=["nombre", "tipo"], name="Efectores de salud (IGN)")

folium.LayerControl().add_to(mapa)
mapa

---

## 6. Geocodificación y creación de capas de puntos

**Conceptos clave.** La **geocodificación** convierte una descripción textual de un lugar —una
dirección, el nombre de una institución— en coordenadas. La operación inversa, la
**geocodificación inversa**, parte de un par de coordenadas y devuelve la dirección postal
correspondiente.

**Escenario.** Es el paso previo a cualquier análisis espacial sobre datos que no nacieron
georreferenciados: las encuestas registran domicilios, los padrones registran direcciones, los
registros administrativos registran nombres de establecimientos. Ninguno de esos datos es
todavía una capa.

El servicio que vamos a usar es **Nominatim**, el geocodificador de OpenStreetMap: gratuito, sin
registro y con un límite de un pedido por segundo.

### 👀 Qué tenemos y qué nos falta

Antes de geocodificar, conviene mirar la capa de efectores de salud y ver con qué información
cuenta realmente.

In [ ]:
print("Columnas:", list(salud.columns))
salud.head()

La capa trae el nombre, el tipo de establecimiento, el barrio y la **geometría**: cada efector
ya está ubicado, porque el IGN lo publicó con sus coordenadas.

Lo que **no** trae es la dirección postal. Y para un informe, para cruzar con un padrón
municipal o para que alguien vaya hasta ahí, la dirección hace falta.

Tenemos entonces el caso inverso al habitual: sobra la coordenada y falta el texto. Es
exactamente lo que resuelve la geocodificación inversa, y como además conservamos el punto
original vamos a poder cerrar el circuito —dirección de vuelta a coordenada— y **medir el error**
que la operación introduce.

### ▶️ Cómo funciona: una sola dirección

Antes de aplicarlo a los 13 registros, conviene ver qué hace la función con un caso.
`geocode()` recibe una cadena de texto y devuelve un objeto con la ubicación encontrada.

In [ ]:
from geopy.geocoders import Nominatim

# El user_agent es obligatorio: identifica a quien consulta el servicio
nominatim = Nominatim(user_agent="sig-unab-2026-clase6")

ubicacion = nominatim.geocode(
    "Avenida Córdoba 2351, Ciudad Autónoma de Buenos Aires, Argentina",
    addressdetails=True)   # pide además la dirección desglosada en campos

print("Dirección normalizada:", ubicacion.address)
print("Latitud:  ", ubicacion.latitude)
print("Longitud: ", ubicacion.longitude)

Tres cosas para observar en el resultado.

La **dirección normalizada** que devuelve no es la que pedimos: es la que el servicio tiene
registrada, con la jerarquía administrativa completa —barrio, comuna, ciudad, código postal,
país—. Sirve para verificar que resolvió lo que queríamos y no otra cosa.

Las **coordenadas** vienen en EPSG:4326, latitud y longitud, que es el estándar de los servicios
web. Para incorporarlas a un GeoDataFrame hay que armar el punto con
`gpd.points_from_xy(lon, lat)`, en ese orden: primero longitud, después latitud.

Y con `addressdetails=True` el objeto trae un diccionario `raw` con la dirección **desglosada en
campos**, que es de donde vamos a sacar la calle y la altura por separado. Sin ese argumento el
servicio devuelve la dirección solo como una cadena de texto.

In [ ]:
ubicacion.raw["address"]

### ▶️ De la coordenada a la dirección, para los 13

`RateLimiter` espera el intervalo que exige el servicio entre un pedido y el siguiente. Sin él,
el servidor bloquea la consulta. Son 13 registros: unos 15 segundos.

In [ ]:
from geopy.extra.rate_limiter import RateLimiter

inverso = RateLimiter(nominatim.reverse, min_delay_seconds=1.1)

respuestas = [inverso((punto.y, punto.x)) for punto in salud.geometry]

salud["calle"]  = [r.raw["address"].get("road") for r in respuestas]
salud["altura"] = [r.raw["address"].get("house_number") for r in respuestas]
salud["tiene_direccion"] = salud["calle"].notna() & salud["altura"].notna()

salud[["barrio", "nombre", "calle", "altura"]]

### ✅ Comprobación

Los 13 devolvieron calle; tres no devolvieron altura. Sin altura la dirección no es
geocodificable de vuelta, así que esos tres quedan fuera del paso siguiente.

La tasa de éxito de una geocodificación se informa junto con el resultado: un análisis que
declara "se geocodificaron 10 de 13 registros" es reproducible.

In [ ]:
print(f"Con calle: {salud['calle'].notna().sum()} de {len(salud)}")
print(f"Con dirección completa: {salud['tiene_direccion'].sum()} de {len(salud)}")

### ▶️ De la dirección a la coordenada

Repetimos la operación del ejemplo individual, ahora sobre las diez direcciones completas. La
consulta incluye barrio, ciudad y país: cuanto más contexto, menor la ambigüedad, porque hay
calles homónimas en muchas localidades del país.

In [ ]:
directo = RateLimiter(nominatim.geocode, min_delay_seconds=1.1)

salud["consulta"] = (salud["calle"] + " " + salud["altura"].astype(str)
                     + ", " + salud["barrio"]
                     + ", Ciudad Autónoma de Buenos Aires, Argentina")

hallados = [directo(c) if tiene else None
            for c, tiene in zip(salud["consulta"], salud["tiene_direccion"])]

salud["lon_geo"] = [r.longitude if r else None for r in hallados]
salud["lat_geo"] = [r.latitude  if r else None for r in hallados]

geocodificados = gpd.GeoDataFrame(
    salud, geometry=gpd.points_from_xy(salud["lon_geo"], salud["lat_geo"]), crs=GEOGRAFICO)

print(f"Resolvió {salud['lat_geo'].notna().sum()} de {salud['tiene_direccion'].sum()}")

### ▶️ El error de la operación

Distancia entre el punto que devolvió Nominatim y el punto del IGN. Es una medición, así que se
hace sobre las capas métricas.

In [ ]:
d = salud_m.geometry.distance(geocodificados.to_crs(METRICO).geometry)
salud["error_m"] = d.where(d.notna()).round()

salud.groupby("barrio")["error_m"].agg(["count", "median", "max"]).round(1)

In [ ]:
fig, eje = plt.subplots(figsize=(9, 8))

barrios_m.plot(ax=eje, facecolor="#f7f7f7", edgecolor="black", linewidth=1.2)
salud_m.plot(ax=eje, color="crimson", markersize=70, label="Original (IGN)")
geocodificados.to_crs(METRICO).plot(ax=eje, color="navy", marker="x", markersize=70,
                                    label="Geocodificado (Nominatim)")

eje.legend(loc="upper right")
eje.set_title("Punto original y punto recuperado desde la dirección")
eje.set_axis_off()
plt.show()

### 🔍 Interpretación

Las diez direcciones completas volvieron a coordenada. El error mediano es de **4,5 metros en
Villa Lugano y 36,5 en Recoleta**, con un máximo de 89. Para variables construidas a escala de
barrio es un error despreciable: en el mapa los pares de símbolos aparecen superpuestos.

La diferencia entre barrios no responde a la calidad del dato sino al tamaño de los
establecimientos. Un CeSAC ocupa un edificio sobre una calle, con una puerta y una altura; el
Hospital de Clínicas ocupa media manzana y tiene entradas por tres calles. Cuanto mayor el
establecimiento, más arbitrario resulta cuál es su punto. Es una limitación del modelo de datos
—se representa con un punto algo que no lo es—, no del geocodificador.

**Criterio operativo:** se geocodifica por dirección, no por nombre institucional. La dirección
es un dato estructurado y estandarizado; el nombre es texto libre, y el servicio solo resuelve
aquellos que alguien cargó con ese nombre exacto.

> 🤖 **Actividad de IA.** Pedile a un asistente la dirección del CeSAC Nº 29 de Villa Lugano y
> compará con la que devolvió la geocodificación inversa. Si coinciden, geocodificá la de la IA
> y medí a cuántos metros cae del punto del IGN.

---

## 7. Transformar geometrías

**Conceptos clave.** Esta familia de operaciones toma una geometría y devuelve otra. No produce
ninguna medida por sí misma: acondiciona la geometría para la operación que sigue.

| Operación | Devuelve | Cuándo se usa |
|---|---|---|
| `representative_point()` | Un punto garantizado interior | Homogeneizar una capa de geometrías mixtas |
| `centroid` | El centro de masa | Medir distancias entre áreas, que no tienen "una" posición |
| `buffer(d)` | La zona a menos de *d* | Definir un área de servicio o de exposición |
| `convex_hull` | La envolvente convexa | Delimitar el área que abarca un conjunto de puntos |
| `envelope` | El rectángulo contenedor | Prefiltrar una capa grande antes de una operación costosa |
| `simplify(t)` | La forma con menos vértices | Publicar un mapa web sin que se trabe |
| `union_all()` | Todo fundido en una geometría | Eliminar superposiciones antes de medir superficie |

**Escenario.** Los dos primeros casos de este bloque son tareas de **preparación de datos**, de
las que hay que hacer casi siempre antes de analizar: homogeneizar los tipos de geometría de una
capa, y recortar una capa nacional a la zona de estudio. Los dos últimos son de publicación y de
medición.

### ▶️ Caso 1 — Homogeneizar tipos de geometría

Al descargar equipamiento de OpenStreetMap no se obtiene una capa de puntos. Un kiosco está
cargado como punto, pero una escuela o un hospital suelen estar cargados como el **polígono del
edificio**, y una parada de colectivos puede ser una línea. La capa mezcla tipos.

In [ ]:
equip_m.geom_type.value_counts()

1.463 puntos, 306 polígonos y una línea. Eso trae dos problemas concretos:

- **Las distancias no son comparables.** `distance` mide hasta el **borde** de un polígono y
  hasta el punto en un punto. Un hospital de una manzana aparece más cerca que un kiosco que
  está a la misma distancia de su centro.
- **La pertenencia se vuelve ambigua.** Un polígono puede quedar a caballo del límite entre dos
  barrios, y entonces no está `within` de ninguno.

`representative_point()` devuelve, para cada geometría, un punto **garantizado interior**. Para
un punto devuelve el mismo punto; para un polígono, uno adentro.

In [ ]:
equip_pt = equip_m.copy()
equip_pt["geometry"] = equip_m.representative_point()

print("Antes:  ", equip_m.geom_type.value_counts().to_dict())
print("Después:", equip_pt.geom_type.value_counts().to_dict())
print("Todos los puntos caen dentro de su geometría original:",
      bool(equip_pt.geometry.within(equip_m.geometry).all()
           or equip_pt.geometry.intersects(equip_m.geometry).all()))

### 👀 El resultado, sobre los polígonos

Acercamiento a un grupo de equipamientos cargados como edificio, con el punto que los reemplaza.

In [ ]:
poligonos = equip_m[equip_m.geom_type.isin(["Polygon", "MultiPolygon"])]
zona = poligonos.geometry.iloc[0].buffer(400).bounds

fig, eje = plt.subplots(figsize=(8, 8))
poligonos.plot(ax=eje, facecolor="#cfe3f5", edgecolor="#2b6ca3", linewidth=1)
equip_pt.loc[poligonos.index].plot(ax=eje, color="crimson", markersize=20)

eje.set_xlim(zona[0], zona[2])
eje.set_ylim(zona[1], zona[3])
eje.set_title("Equipamientos cargados como polígono, y su punto representativo")
eje.set_axis_off()
plt.show()

De acá en adelante trabajamos con `equip_pt`: una capa de 1.770 puntos, homogénea, donde cada
equipamiento pesa lo mismo sin importar cómo lo cargó quien lo mapeó.

El efecto sobre las mediciones es acotado pero real: en los 306 equipamientos que eran
polígonos, la distancia al efector de salud más cercano aumenta en promedio unos 28 metros al
medirla desde el punto interior y no desde el borde del edificio.

### ▶️ Caso 2 — Recortar una capa nacional a la zona de estudio

La capa de salud del IGN cubre todo el país. Nuestra zona de estudio son dos barrios. Antes de
cualquier operación conviene quedarse solo con lo que puede llegar a interesar.

In [ ]:
salud_pais = gpd.read_file(DATOS + "ign_salud.gpkg").to_crs(METRICO)

print(f"{len(salud_pais)} establecimientos de salud en todo el país")

In [ ]:
provincias = gpd.read_file(DATOS + "provincias_arg.gpkg").to_crs(METRICO)

fig, eje = plt.subplots(figsize=(6, 9))
provincias.boundary.plot(ax=eje, color="grey", linewidth=0.5)
salud_pais.plot(ax=eje, color="crimson", markersize=1, alpha=0.5)

eje.set_title("Capa nacional de salud del IGN — 8.311 establecimientos")
eje.set_axis_off()
plt.show()

El recorte se puede hacer con tres geometrías distintas, cada una más ajustada y más costosa de
calcular que la anterior:

- el **rectángulo contenedor** (`envelope`) de los dos barrios,
- su **envolvente convexa** (`convex_hull`),
- y la **geometría exacta** de los barrios, disuelta.

In [ ]:
zona_estudio = barrios_m.union_all()

recortes = {
    "envelope":    zona_estudio.envelope,
    "convex_hull": zona_estudio.convex_hull,
    "exacta":      zona_estudio,
}

pd.DataFrame([
    {"filtro": nombre,
     "superficie_km2": round(g.area / 1e6, 2),
     "establecimientos": int(salud_pais.intersects(g).sum())}
    for nombre, g in recortes.items()
]).set_index("filtro")

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 6))

for eje, (nombre, geom) in zip(ejes, recortes.items()):
    gpd.GeoSeries([geom], crs=METRICO).plot(ax=eje, facecolor="#ffe9c9",
                                            edgecolor="#d98c00", linewidth=1.5)
    barrios_m.plot(ax=eje, facecolor="none", edgecolor="black", linewidth=1.2)
    dentro = salud_pais[salud_pais.intersects(geom)]
    dentro.plot(ax=eje, color="crimson", markersize=25)
    eje.set_title(f"{nombre} — {len(dentro)} establecimientos")
    eje.set_axis_off()

plt.tight_layout()
plt.show()

### 🔍 Interpretación

De 8.311 establecimientos, el rectángulo deja **122**, la envolvente convexa **46** y la
geometría exacta **13**.

Los tres filtros responden preguntas distintas y se usan en momentos distintos:

- El **`envelope`** es una comparación de cuatro números por punto: descarta el 98,5 % de la capa
  casi sin costo. Es lo que conviene aplicar primero cuando la capa de origen es grande.
- El **`convex_hull`** ajusta más, pero sigue incluyendo todo el espacio entre los dos barrios,
  que no pertenece a ninguno. Su uso propio no es este: sirve para delimitar el área que abarca
  un conjunto disperso de puntos —el alcance de una red, la extensión de una muestra—, donde no
  hay un polígono de referencia.
- La **geometría exacta** es la que da el resultado correcto, y es la más costosa. Aplicada
  después del `envelope` trabaja sobre 122 puntos en lugar de 8.311.

Los 13 que quedan son los de `salud_barrios.gpkg`, el archivo que ya veníamos usando: así se
preparó.

### ▶️ Caso 3 — `centroid`: reducir un área a un punto para medirla

Un polígono no tiene una posición: tiene una extensión. Para medir la distancia entre dos áreas
hay que elegir un punto que las represente, y hay dos candidatos que no son equivalentes.

In [ ]:
centroides      = barrios_m.centroid                   # centro de masa
puntos_internos = barrios_m.representative_point()     # un punto cualquiera, pero interior

pd.DataFrame({
    "barrio": barrios_m["barrio"],
    "centroide_adentro": centroides.within(barrios_m.geometry),
    "punto_interno_adentro": puntos_internos.within(barrios_m.geometry),
})

In [ ]:
fig, eje = plt.subplots(figsize=(7, 7))

barrios_m.plot(ax=eje, facecolor="#eef3f8", edgecolor="#2b6ca3", linewidth=1.5)
centroides.plot(ax=eje, color="crimson", markersize=90, label="centroid")
puntos_internos.plot(ax=eje, color="darkgreen", marker="x", markersize=90,
                     label="representative_point")

eje.legend(loc="upper right")
eje.set_title("Dos maneras de reducir un polígono a un punto")
eje.set_axis_off()
plt.show()

En estos dos barrios ambos puntos caen adentro, porque son formas compactas. El centroide es el
centro de masa y nada garantiza que esté dentro de la figura: en un polígono cóncavo —una
provincia con una bahía, un partido con forma de herradura— cae fuera. De ahí la diferencia de
uso: para rotular o para muestrear, `representative_point()`; para medir distancias entre áreas,
el centroide.

El centroide de un barrio **no representa a su población**: es una simplificación geométrica, no
demográfica.

### ▶️ Caso 4 — `simplify`: aligerar para publicar

Un mapa interactivo con geometrías detalladas tarda en cargar y puede trabar el navegador. Es lo
que hicimos en la Clase 5 con los 527 departamentos. `simplify` reduce la cantidad de vértices
con una tolerancia expresada en las unidades del CRS: cuánto se admite que la línea nueva se
aparte de la original.

In [ ]:
simplificado = barrios_m.simplify(200)

pd.DataFrame({
    "vertices": [int(barrios_m.count_coordinates().sum()),
                 int(simplificado.count_coordinates().sum())],
    "superficie_km2": [round(barrios_m.area.sum() / 1e6, 2),
                       round(simplificado.area.sum() / 1e6, 2)],
}, index=["original", "simplificado 200 m"])

In [ ]:
fig, eje = plt.subplots(figsize=(9, 7))

barrios_m.plot(ax=eje, facecolor="none", edgecolor="black", linewidth=1.6)
simplificado.plot(ax=eje, facecolor="none", edgecolor="crimson",
                  linewidth=1.6, linestyle="--")

eje.set_title("Original (negro) y simplificado con tolerancia de 200 m (rojo)")
eje.set_axis_off()
plt.show()

De 797 vértices a 40 —un 95 % menos— con una pérdida del 3 % de superficie. Las curvas suaves se
convierten en rectas y el detalle menor a 200 metros desaparece. Para un mapa de conjunto es un
intercambio conveniente; para medir superficies no lo es, y la medición se hace siempre sobre la
geometría original.

---

## 8. Áreas de influencia, disolución y superposición

**Conceptos clave.** Un **área de influencia** (*buffer*) es la zona que rodea a una geometría
hasta una distancia dada. La **superposición** (*overlay*) corta una capa contra otra:
`intersection` devuelve la parte común y `difference` la parte que queda fuera. Entre las dos
operaciones hay un paso necesario, la **disolución** (`union_all`), que funde varias geometrías
en una sola y elimina las superposiciones.

**Escenario.** Queremos analizar la cobertura de los efectores públicos de salud. Suponemos que
un efector es accesible a pie para quien vive a una distancia razonable: **500 metros**, unos
seis minutos de caminata, el umbral que suele usarse para servicios de proximidad. La pregunta
es qué parte de cada barrio queda dentro de ese radio y qué parte queda fuera.

Los 500 metros son un supuesto, no un dato. Al final del bloque medimos cuánto depende el
resultado de esa elección.

### ▶️ Paso 1 — Las áreas de influencia

`buffer` interpreta el número en las unidades del CRS de la capa. Por eso se aplica sobre
`salud_m`, que está en metros.

In [ ]:
RADIO_M = 500

areas = salud_m.copy()
areas["geometry"] = salud_m.buffer(RADIO_M)

print(f"{len(areas)} áreas de influencia de {RADIO_M} m")
print(f"Superficie de cada una: {areas.area.iloc[0] / 10_000:.1f} ha")

In [ ]:
fig, eje = plt.subplots(figsize=(9, 8))

barrios_m.plot(ax=eje, facecolor="#f2f2f2", edgecolor="black", linewidth=1.5)
areas.plot(ax=eje, facecolor="orange", alpha=0.35, edgecolor="darkorange", linewidth=0.8)
salud_m.plot(ax=eje, color="crimson", markersize=35)

eje.set_title(f"Áreas de influencia de {RADIO_M} m alrededor de cada efector")
eje.set_axis_off()
plt.show()

El mapa muestra el problema del paso siguiente. En Villa Lugano los círculos **se superponen**:
hay zonas alcanzadas por dos o tres efectores a la vez. Si sumáramos las superficies de los 13
círculos, esas zonas se contarían varias veces y el porcentaje de cobertura podría superar el
100 %.

### ▶️ Paso 2 — Disolver

`union_all()` funde las 13 geometrías en una y elimina las superposiciones. El resultado ya no
es una capa de 13 registros sino **una única geometría**, la zona de cobertura del conjunto.

Es un `MultiPolygon` y no un `Polygon` porque los círculos de Recoleta y los de Villa Lugano no
llegan a tocarse: la geometría tiene dos partes separadas, pero es una sola.

In [ ]:
cobertura = salud_m.buffer(RADIO_M).union_all()

print("Tipo de geometría:", cobertura.geom_type)
print(f"Suma de los 13 círculos por separado: {areas.area.sum() / 1e6:.2f} km²")
print(f"Superficie de la zona disuelta:       {cobertura.area / 1e6:.2f} km²")
print(f"Diferencia (superposición):           {(areas.area.sum() - cobertura.area) / 1e6:.2f} km²")

### 👀 La zona disuelta

Éste es el objeto con el que se trabaja de acá en adelante. Conviene mirarlo antes de usarlo,
porque es el insumo de todas las mediciones del bloque.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 7))

areas.plot(ax=ejes[0], facecolor="orange", alpha=0.45,
           edgecolor="darkorange", linewidth=1)
ejes[0].set_title(f"Antes: 13 áreas superpuestas ({areas.area.sum() / 1e6:.2f} km²)")

gpd.GeoSeries([cobertura], crs=METRICO).plot(ax=ejes[1], facecolor="orange", alpha=0.45,
                                             edgecolor="darkorange", linewidth=1)
ejes[1].set_title(f"Después: una sola geometría ({cobertura.area / 1e6:.2f} km²)")

for eje in ejes:
    barrios_m.plot(ax=eje, facecolor="none", edgecolor="black", linewidth=1.2)
    salud_m.plot(ax=eje, color="crimson", markersize=25)
    eje.set_axis_off()

plt.tight_layout()
plt.show()

A la izquierda se distinguen los bordes de cada círculo dentro de cada grupo; a la derecha esos
contornos internos desaparecieron. La diferencia de 1,88 km² entre ambas superficies —un 18 %
del total— es exactamente lo que se habría contado dos veces al sumar los círculos por separado.

### ▶️ Paso 3 — Intersección y diferencia

Con la zona de cobertura disuelta, la superposición contra cada barrio devuelve las dos partes:
la cubierta y la que queda fuera.

In [ ]:
barrios_m["superficie_km2"] = barrios_m.area / 1e6
barrios_m["area_cubierta_km2"] = barrios_m.geometry.intersection(cobertura).area / 1e6
barrios_m["cobertura_500m_perc"] = (barrios_m["area_cubierta_km2"]
                                    / barrios_m["superficie_km2"] * 100)

barrios_m[["barrio", "superficie_km2", "area_cubierta_km2", "cobertura_500m_perc"]].round(2)

In [ ]:
cubierto = barrios_m.copy()
cubierto["geometry"] = barrios_m.geometry.intersection(cobertura)

afuera = barrios_m.copy()
afuera["geometry"] = barrios_m.geometry.difference(cobertura)

fig, eje = plt.subplots(figsize=(9, 8))
cubierto.plot(ax=eje, facecolor="#7fbf7b", edgecolor="none")
afuera.plot(ax=eje, facecolor="#e08a8a", edgecolor="none")
barrios_m.plot(ax=eje, facecolor="none", edgecolor="black", linewidth=1.5)
salud_m.plot(ax=eje, color="darkred", markersize=30)

eje.set_title(f"A menos de {RADIO_M} m de un efector (verde) y a más (rojo)")
eje.set_axis_off()
plt.show()

### ✅ Comprobación

`intersection` y `difference` son complementarias: sus superficies tienen que reconstruir el
barrio completo. Si no lo hacen, se perdió geometría en el camino.

In [ ]:
reconstruido = (cubierto.area + afuera.area) / 1e6

pd.DataFrame({"reconstruido_km2": reconstruido.round(4),
              "original_km2": barrios_m["superficie_km2"].round(4),
              "barrio": barrios_m["barrio"]}).set_index("barrio")

### ▶️ Paso 4 — Sensibilidad al umbral

El radio de 500 metros fue un supuesto. Antes de concluir, conviene medir cuánto cambia el
resultado si ese supuesto cambia.

In [ ]:
filas = []
for radio in [200, 300, 500, 750, 1000, 1500]:
    zona = salud_m.buffer(radio).union_all()
    for _, b in barrios_m.iterrows():
        filas.append(dict(radio_m=radio, barrio=b["barrio"],
                          cobertura=100 * b.geometry.intersection(zona).area / b.geometry.area))

sensibilidad = pd.DataFrame(filas).pivot(index="radio_m", columns="barrio", values="cobertura")
sensibilidad.round(1)

In [ ]:
eje = sensibilidad.plot(marker="o", figsize=(8, 5), color=["#c44e52", "#4c72b0"])
eje.set_xlabel("Radio del área de influencia (m)")
eje.set_ylabel("% de la superficie del barrio cubierta")
eje.set_title("Cobertura según el umbral elegido")
eje.grid(linestyle=":", alpha=0.6)
eje.legend(title="")
plt.tight_layout()
plt.show()

### 🔍 Interpretación

A 500 metros, **Villa Lugano tiene el 49 % de su superficie cubierta y Recoleta el 24 %**.

El resultado se explica por la composición de la oferta, no por su cantidad. Recoleta tiene
cuatro efectores públicos y los cuatro son hospitales de alta complejidad, que atienden a toda
la ciudad y están dimensionados para eso. Villa Lugano tiene nueve, y son CeSAC: centros de
atención primaria, de escala barrial, localizados deliberadamente para dar cobertura de
proximidad.

El gráfico de sensibilidad es lo que permite sostener la conclusión: las dos curvas no se cruzan
en ningún umbral entre 200 y 1.500 metros. El resultado no es un artefacto del radio elegido.

Dos limitaciones que corresponde declarar. La cobertura es de **superficie**, no de población:
el barrio no tiene sus habitantes distribuidos de manera uniforme, y para calcularla sobre
población hace falta el dato por radio censal, que es tema de la Clase 7. Y la distancia es una
dimensión del acceso, no la única: no dice nada sobre horarios, turnos disponibles ni
complejidad de la atención.

---

## 9. Uniones espaciales 1 a 1 y 1 a muchos

**Conceptos clave.** Una **unión espacial** (`gpd.sjoin`) combina los atributos de dos capas a
partir de su relación espacial, sin necesidad de que compartan ninguna columna. La relación se
declara con el **predicado**: `within` (estrictamente adentro), `intersects` (se tocan o se
superponen) o `contains`.

Importa la cardinalidad:

- **1 a 1**: cada elemento de A se corresponde con uno de B. Por ejemplo, asignar a cada punto
  el polígono que lo contiene.
- **1 a muchos**: un polígono contiene varios puntos. La unión duplica el polígono una vez por
  punto, y después se agrega con `groupby`.

**Escenario.** Dos preguntas. Primero, en qué barrio se encuentra cada efector de salud, que es
la operación con la que se preparó `salud_barrios.gpkg` a partir de la capa nacional. Después,
cuánto equipamiento tiene cada barrio y de qué tipo, que es una unión 1 a muchos con agregación.

### ▶️ Unión 1 a 1 — en qué barrio cae cada efector

Retomamos los 122 candidatos que el `envelope` del bloque 7 dejó en pie. El recorte por
rectángulo dijo cuáles **pueden** estar en la zona; la unión espacial dice en cuál de los dos
barrios está cada uno, y descarta los que caen fuera.

In [ ]:
candidatos = salud_pais[salud_pais.intersects(recortes["envelope"])]

efectores_en_barrio = gpd.sjoin(candidatos, barrios_m[["barrio", "geometry"]],
                                predicate="within").drop(columns="index_right")

print(f"Candidatos dentro del rectángulo: {len(candidatos)}")
print(f"Efectivamente dentro de un barrio: {len(efectores_en_barrio)}")
efectores_en_barrio["barrio"].value_counts()

Cuatro en Recoleta y nueve en Villa Lugano: los mismos 13 de `salud_barrios.gpkg`. Cada punto
incorporó la columna `barrio` del polígono que lo contiene, que es todo lo que hace una unión
espacial 1 a 1: **transferir al punto el atributo del área donde está**.

Es la operación que asigna una encuesta a su radio censal, un hecho registrado a su comuna o un
establecimiento a su partido.

### ▶️ Unión 1 a muchos — cuánto equipamiento tiene cada barrio

Usamos `equip_pt`, la capa homogeneizada a puntos en el bloque 7. La capa trae además una
columna `barrio`, de cuando se descargó un barrio por vez en la Clase 3: se descarta antes de la
unión, porque si quedaran las dos GeoPandas renombraría ambas como `barrio_left` y
`barrio_right`.

In [ ]:
equip_pt = equip_pt.drop(columns=["barrio"])

equip_en_barrio = gpd.sjoin(equip_pt, barrios_m[["barrio", "geometry"]],
                            predicate="within").drop(columns="index_right")

print(f"De {len(equip_pt)} equipamientos, {len(equip_en_barrio)} quedaron asignados")

Uno de los 1.770 quedó sin asignar: cae exactamente sobre el límite. `within` exige estar
estrictamente adentro; con `intersects` habría entrado, y en una zona de estudio con muchas
unidades vecinas habría entrado en dos a la vez. La elección del predicado es una decisión del
análisis, no un detalle.

### 👀 Verificación de la unión

Cada punto pintado con el color del barrio que le asignó el `sjoin`. Un punto de un color dentro
del polígono del otro indicaría un problema de sistema de referencia o de predicado.

In [ ]:
fig, eje = plt.subplots(figsize=(9, 8))

barrios_m.plot(ax=eje, facecolor="none", edgecolor="black", linewidth=1.5)
equip_en_barrio.plot(ax=eje, column="barrio", markersize=6, legend=True,
                     cmap="Set1", legend_kwds=dict(loc="upper right"))

eje.set_title("Cada equipamiento, con el barrio que le asignó el sjoin")
eje.set_axis_off()
plt.show()

### ▶️ Agregación

La unión duplicó cada barrio una vez por equipamiento contenido. `groupby` lo resume a un valor
por unidad de análisis, que es lo que la tabla final necesita.

In [ ]:
conteo = equip_en_barrio.groupby("barrio").size().rename("equipamientos")

por_tipo = equip_en_barrio.pivot_table(index="amenity", columns="barrio",
                                       aggfunc="size", fill_value=0)

por_tipo.sort_values("Recoleta", ascending=False).head(10)

### ▶️ De conteo a densidad

Los conteos no son comparables entre unidades de distinto tamaño: Recoleta tiene 6,9 km² y
Villa Lugano 9,3. Es la distinción de la Clase 5, ahora aplicada a la producción del dato.

In [ ]:
densidad = por_tipo.T.join(barrios_m.set_index("barrio")["superficie_km2"])
for tipo in ["pharmacy", "school", "cafe", "bank", "clinic"]:
    densidad[tipo + "_por_km2"] = (densidad[tipo] / densidad["superficie_km2"]).round(1)

densidad[[c for c in densidad.columns if c.endswith("_por_km2")]]

In [ ]:
columnas = [c for c in densidad.columns if c.endswith("_por_km2")]
grafico = densidad[columnas].rename(columns=lambda c: c.replace("_por_km2", ""))

eje = grafico.T.plot.barh(figsize=(9, 5), color=["#c44e52", "#4c72b0"], width=0.75)
eje.set_xlabel("Equipamientos por km²")
eje.set_ylabel("")
eje.set_title("Densidad de equipamiento por barrio")
eje.legend(title="")
eje.grid(axis="x", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

### 🔍 Interpretación

Las barras se ordenan en dos grupos. **Farmacias: 12,4 por km² en Recoleta contra 0,8 en Villa
Lugano.** Bancos: 8,2 contra 0,6. Cafés: 31,4 contra 0,4. **Escuelas: 9,2 y 9,2.**

Farmacias, bancos y cafés son equipamiento de mercado y se localizan según la capacidad de
compra. Las escuelas son equipamiento público y se localizan según la población en edad
escolar. Son dos lógicas de localización distintas y dejan huellas territoriales distintas.

La fila de las escuelas cumple además una función de control sobre la fuente: si OpenStreetMap
subregistrara Villa Lugano de manera sistemática, las escuelas también aparecerían subcontadas.

---

## 10. Centroides y distancias

**Conceptos clave.** La **distancia euclídea** entre dos geometrías es la separación mínima en
línea recta. `sjoin_nearest` resuelve en una sola operación cuál es el elemento más próximo de
otra capa y a qué distancia está, lo que a mano exigiría comparar todos contra todos.

**Escenario.** La cobertura del bloque 8 responde con un sí o un no: dentro o fuera de los 500
metros. La distancia responde con un número, y por eso admite comparaciones más finas. Vamos a
calcular, para cada equipamiento del barrio, a qué distancia está el efector de salud más
cercano, y a resumir ese valor por barrio.

La distancia es **en línea recta**. La distancia efectiva por la red de calles es siempre mayor,
y la calculamos en la Clase 7.

### ▶️ Distancia al efector más cercano

In [ ]:
cercano = gpd.sjoin_nearest(
    equip_en_barrio,
    salud_m[["nombre", "geometry"]].rename(columns={"nombre": "efector_cercano"}),
    distance_col="dist_salud_m",
).drop(columns="index_right")

cercano[["name", "amenity", "barrio", "efector_cercano", "dist_salud_m"]].head()

### 👀 Verificación

Si la operación es correcta, los puntos próximos a un efector deben ser claros y los alejados
oscuros: alrededor de cada efector tiene que verse un halo.

In [ ]:
fig, eje = plt.subplots(figsize=(10, 8))

barrios_m.plot(ax=eje, facecolor="#f7f7f7", edgecolor="black", linewidth=1.5)
cercano.plot(ax=eje, column="dist_salud_m", cmap="YlOrRd", markersize=14, legend=True,
             legend_kwds=dict(label="Distancia al efector más cercano (m)", shrink=0.6))
salud_m.plot(ax=eje, color="black", marker="P", markersize=90)

eje.set_title("Equipamientos según su distancia al efector de salud más cercano")
eje.set_axis_off()
plt.show()

### ▶️ Resumen por unidad de análisis

`sjoin_nearest` devolvió una distancia por cada equipamiento. La tabla final necesita un valor
por barrio.

In [ ]:
resumen_dist = (cercano.groupby("barrio")["dist_salud_m"]
                .agg(["count", "mean", "median", "max"]).round(0))
resumen_dist

In [ ]:
fig, eje = plt.subplots(figsize=(9, 5))

for barrio, color in [("Recoleta", "#c44e52"), ("Villa Lugano", "#4c72b0")]:
    datos = cercano.loc[cercano["barrio"] == barrio, "dist_salud_m"]
    eje.hist(datos, bins=30, alpha=0.6, label=barrio, color=color, density=True)

eje.axvline(RADIO_M, color="black", linestyle="--", linewidth=1)
eje.text(RADIO_M + 20, eje.get_ylim()[1] * 0.9, f"{RADIO_M} m", fontsize=9)
eje.set_xlabel("Distancia al efector de salud más cercano (m)")
eje.set_ylabel("Densidad")
eje.set_title("Distribución de la distancia, por barrio")
eje.legend()
plt.tight_layout()
plt.show()

### ▶️ Desagregado por tipo de equipamiento

In [ ]:
(cercano[cercano["amenity"].isin(["school", "pharmacy", "kindergarten", "cafe"])]
 .pivot_table(index="amenity", columns="barrio", values="dist_salud_m", aggfunc="median")
 .round(0))

### 🔍 Interpretación

La distancia mediana de un equipamiento al efector de salud más cercano es de **422 metros en
Villa Lugano y 604 en Recoleta**. El resultado coincide con el del bloque 8, obtenido por otra
vía: dos operaciones independientes sobre las mismas capas conducen a la misma conclusión.

El histograma agrega información que el promedio no contiene. La distribución de Villa Lugano
es angosta y está corrida hacia la izquierda —casi todo su equipamiento se encuentra entre 200
y 700 metros de un CeSAC—, mientras que la de Recoleta tiene una cola que llega a 2,6 km.

El desagregado por tipo es el que admite una lectura sectorial: **las escuelas de Villa Lugano
están a 353 metros medianos de un efector de salud y las de Recoleta a 564**. Para dimensionar
un programa de salud escolar, ése es el valor pertinente y no el promedio general.

---

## 11. La tabla de variables territoriales

### ▶️ Composición

Una fila por unidad de análisis, una columna por variable, la unidad de medida en el nombre.

In [ ]:
tabla = barrios_m[["barrio", "superficie_km2", "cobertura_500m_perc"]].copy()

tabla["efectores_salud"]   = tabla["barrio"].map(salud["barrio"].value_counts())
tabla["equipamientos"]     = tabla["barrio"].map(conteo)
tabla["farmacias_por_km2"] = tabla["barrio"].map(densidad["pharmacy_por_km2"])
tabla["escuelas_por_km2"]  = tabla["barrio"].map(densidad["school_por_km2"])
tabla["dist_salud_m"]      = tabla["barrio"].map(resumen_dist["median"])

tabla.round(1)

### ▶️ Exportación

El GeoPackage conserva la geometría y permite continuar el trabajo; el CSV es la tabla para el
informe.

In [ ]:
salida = barrios[["barrio", "geometry"]].merge(tabla, on="barrio")

salida.to_file("variables_territoriales_barrios.gpkg", driver="GPKG")
tabla.round(2).to_csv("variables_territoriales_barrios.csv", index=False)

print("variables_territoriales_barrios.gpkg — con geometría")
print("variables_territoriales_barrios.csv  — la tabla")

En Colab estos archivos quedan en el disco temporal de la sesión y se borran al cerrarla. Para
conservarlos hay que descargarlos desde el panel de archivos o montar Google Drive.

---

## 12. 🧪 Actividad

Repetir el análisis cambiando el equipamiento de referencia: en lugar de los efectores de salud
del IGN, usar las **escuelas** de OpenStreetMap, que presentaron densidad idéntica en los dos
barrios.

El esqueleto de las operaciones está resuelto. **El mapa no**: completalo, porque sin la
verificación visual no hay manera de saber si el resultado es el que corresponde.

In [ ]:
# 1 · Equipamiento de referencia y radio
TIPO  = "school"      # alternativas: "pharmacy", "kindergarten", "bank"
RADIO = 500           # metros

referencia = equip_en_barrio[equip_en_barrio["amenity"] == TIPO]

# 2 · Cobertura: buffer, disolución, intersección
zona = referencia.buffer(RADIO).union_all()
resultado = barrios_m[["barrio", "superficie_km2"]].copy()
resultado["cubierto_perc"] = [round(100 * g.intersection(zona).area / g.area, 1)
                              for g in barrios_m.geometry]

# 3 · Distancia al más cercano
otros = equip_en_barrio[equip_en_barrio["amenity"] != TIPO]
dist = gpd.sjoin_nearest(otros, referencia[["geometry"]], distance_col="dist_m")
resultado["dist_mediana_m"] = resultado["barrio"].map(
    dist.groupby("barrio")["dist_m"].median().round(0))

resultado

In [ ]:
# 4 · Completar: la zona disuelta, la parte cubierta y la que queda fuera
fig, eje = plt.subplots(figsize=(9, 8))

# ...

eje.set_title(f"Cobertura de {TIPO} a {RADIO} m")
eje.set_axis_off()
plt.show()

**Preguntas, para responder por escrito:**

1. ¿Qué barrio queda mejor cubierto? ¿El resultado coincide con la densidad por km² del bloque
   9? Si no coincide, ¿qué puede explicarlo?
2. Repetí el cálculo con radios de 300 y 1.000 metros y rehacé el gráfico de sensibilidad del
   bloque 8. ¿Las curvas se cruzan? ¿Qué implica eso para la conclusión?
3. ¿Cobertura y distancia dan la misma respuesta? Si no coinciden, hay algo que explicar.
4. Observá el mapa que construiste. ¿Alguna zona cubierta resulta inesperada respecto de lo que
   sabés del barrio?

**Entrega:** media carilla, la tabla de resultados y los dos mapas, antes del próximo encuentro.

---

## 13. 📋 Trabajo final

Las cuatro familias de operaciones de esta clase —transformar, superponer, relacionar y medir—
son las herramientas con las que se construyen las variables del trabajo final.

La consigna completa está en
[`TRABAJO_FINAL.md`](https://github.com/renzoepolo/sig-ciencias-sociales/blob/main/TRABAJO_FINAL.md)
y en el aula virtual: una notebook de Colab que corra de principio a fin, con una pregunta
territorial propia, al menos dos fuentes abiertas, dos variables territoriales construidas, dos
mapas terminados y una sección de limitaciones.

Para la Clase 7 hay que traer definida la **pregunta y la unidad de análisis**, y verificado que
existan los datos.

---

## 14. Cierre

### Recorrido

| Familia | Operación | Pregunta que responde | Variable resultante |
|---|---|---|---|
| — | Geocodificación | ¿Cómo convierto direcciones en una capa? | — |
| **Transformar** | `centroid`, `convex_hull`, `envelope`, `simplify`, `union_all` | ¿Qué forma necesito para la medición que quiero hacer? | — |
| **Superponer** | `intersection`, `difference` | ¿Qué parte de la unidad está dentro de la zona? | `cobertura_500m_perc` |
| **Relacionar** | `sjoin` + `groupby` | ¿Qué contiene cada unidad? | `farmacias_por_km2` |
| **Medir** | `sjoin_nearest` | ¿A qué distancia está lo más próximo? | `dist_salud_m` |

### Conclusión de la clase

Una variable territorial no se observa: se construye, y cada una incorpora decisiones que no
quedan registradas en el nombre de la columna. El radio de 500 metros, el predicado `within`, la
opción de normalizar por superficie en lugar de por población: ninguna figura en
`cobertura_500m_perc`, y todas modifican el valor.

De ahí la verificación visual en cada paso. Dos veces en esta clase el gráfico aportó lo que la
tabla no mostraba: el halo de `sjoin_nearest`, que confirma que la operación hizo lo que
declaraba, y las curvas de sensibilidad que no se cruzan, que es lo que convierte una
observación en una conclusión sostenible.

### Glosario

| Término | Definición |
|---|---|
| **Operación espacial** | Procedimiento que toma una o más geometrías y devuelve una geometría, una relación o una medida, calculada a partir de su forma y su posición. |
| **Variable territorial** | Atributo de una unidad del espacio obtenido de la relación entre dos o más capas. |
| **Geocodificación** | Conversión de una dirección en coordenadas. La inversa parte de la coordenada. |
| **Centroide** | Centro de masa de una geometría. No necesariamente interior a ella. |
| **Área de influencia** (*buffer*) | Zona que rodea a una geometría hasta una distancia dada. |
| **Envolvente convexa** (*convex hull*) | Polígono convexo mínimo que contiene a la geometría. |
| **Simplificación** | Reducción de vértices conservando la forma general, con una tolerancia declarada. |
| **Disolución** (`union_all`) | Fusión de varias geometrías en una, eliminando superposiciones. |
| **Superposición** (*overlay*) | Corte de una capa contra otra: `intersection`, `difference`, `union`. |
| **Unión espacial** (`sjoin`) | Combinación de atributos de dos capas según su relación espacial. |
| **Predicado espacial** | Relación que la unión espacial evalúa: `within`, `intersects`, `contains`. |
| **Cobertura** | Porcentaje de la superficie de una unidad contenido en un área de influencia. |

### Autoevaluación

1. El porcentaje de cobertura de una unidad resultó 137 %. ¿Qué paso se omitió?
2. Se quiere medir la distancia de cada radio censal al hospital más cercano. ¿Qué
   transformación de geometría se requiere primero, y qué supuesto introduce?
3. Un `sjoin` con `within` dejó 40 de 1.000 puntos sin asignar. ¿Cuáles son las dos
   explicaciones posibles y cómo se distinguen?

### Clase 7

Las distancias de hoy son en línea recta. La distancia efectiva por la red de calles es mayor, y
la próxima clase mide cuánto: vamos a comparar el área de influencia circular de 500 metros con
la superficie realmente alcanzable caminando en ese tiempo.

También resolvemos el problema que quedó abierto en el bloque 8: cómo transferir una variable de
una división territorial a otra —de radios censales a barrios— para calcular la cobertura sobre
población y no sobre superficie.

---

## 15. Referencias

- de Smith, M. J., Goodchild, M. F. y Longley, P. A. (2018). *Geospatial Analysis: A
  Comprehensive Guide to Principles, Techniques and Software Tools* (6.ª ed.), caps. 4 y 7.
- Rey, S., Arribas-Bel, D. y Wolf, L. J. (2023). *Geographic Data Science with Python*, cap. 8
  "Spatial Feature Engineering". CRC Press.
- Instituto Geográfico Nacional (2024). *Capas de Sistema de Información Geográfica — Salud*.
  Geoservicio WFS, capa `ign:salud_020801`.
- Ministerio de Salud del Gobierno de la Ciudad de Buenos Aires. *Centros de Salud y Acción
  Comunitaria (CeSAC)*.
- Documentación de Shapely: predicados binarios y operaciones de conjunto.
  https://shapely.readthedocs.io/en/stable/manual.html